# Faruq-v3 — AF2_ORIENT isolated seed42

Uji **AF2 + unsigned orientation 180° saja**. Semua pilihan AF2 lain dipertahankan: RGB independen, hard suppression, patch 32, overlap 0.5, gamma 0.1, tanpa radial split. Resolusi sudut dijaga ~1°/bin dengan 180 bin untuk periode 180°. Tidak menggunakan luminance gate, soft selection, Hann window, atau perubahan YOLO lain.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import importlib, json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path
import torch

assert torch.cuda.is_available(), 'Aktifkan GPU: Runtime > Change runtime type > T4 GPU.'
REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/af2-isolated-radial-orientation'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone = ['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(1,4):
    result = subprocess.run(clone)
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 3: raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)], check=True)
for name in list(sys.modules):
    if name == 'coffee_detector' or name.startswith('coffee_detector.'):
        sys.modules.pop(name, None)
sys.path.insert(0, str(REPO/'src'))
importlib.invalidate_caches()
os.chdir(REPO)
import ultralytics
assert ultralytics.__version__ == '8.4.96', ultralytics.__version__
print('GPU:', torch.cuda.get_device_name(0))
print('Branch:', BRANCH)


In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
D0 = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
GROUPED = DATA_ROOT/'faruq_grouped_summary.json'
if not GROUPED.is_file():
    if DATA_ROOT.exists(): shutil.rmtree(DATA_ROOT)
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content', filter='data')
assert (DATA_ROOT/'data.yaml').is_file()
assert GROUPED.is_file()
assert not (DATA_ROOT/'test').exists(), 'STOP: test tidak boleh tersedia.'
OUTPUT = PROJECT_ROOT/'experiments/faruq-v3-af2-isolated-seed42-v1'
OUTPUT.mkdir(parents=True, exist_ok=True)
print('D0:', D0)
print('OUTPUT:', OUTPUT)


In [ ]:
ARM = 'AF2_ORIENT'
command = [sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_iso_arm',
           '--arm',ARM,'--data-root',str(DATA_ROOT),'--grouped-summary',str(GROUPED),
           '--d0-checkpoint',str(D0),'--output-root',str(OUTPUT),'--seed','42',
           '--device','0','--latency-iterations','50','--authorize-training']
print('MENJALANKAN:', ' '.join(command), flush=True)
subprocess.run(command, cwd=REPO, check=True)


In [ ]:
RESULT = OUTPUT/'val_reports/AF2_ORIENT_seed42_result.json'
assert RESULT.is_file(), RESULT
payload = json.loads(RESULT.read_text())
m = payload['metrics']
parent = {'macro_map50_95':0.8819734111,'bottom3_class_map50_95':0.800428,'worst_class_map50_95':0.793470}
print('AF2_ORIENT seed42 final:')
for key in parent:
    value = m[key]
    print(f'{key}: {value:.6f} | delta vs AF2 parent ≈ {(value-parent[key])*100:+.3f} pp')
print('Latency median:', payload['latency']['median_ms'], 'ms')
print('Test accessed:', payload['test_images_accessed'])
print('RESULT:', RESULT)
